In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Importing Necessary Libraries

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
!pip install contractions
import contractions
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt_tab')
import string
from sklearn.model_selection import train_test_split
!pip install gensim
from gensim.models import Word2Vec
import logging
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
!pip install tensorflow
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input , Embedding , LSTM , Bidirectional , GRU , Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
import json

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [contractions]


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 52.2 MB/s  0:00:00


In [5]:
BASE_PATH = "/content/drive/MyDrive/smart blind stick"

In [6]:
df = pd.read_csv("/content/drive/MyDrive/smart blind stick/SIT_navigation_intent_dataset.csv")

# Exploring the dataset

In [7]:
df.head(5)

,id,text,intent
0,1,"Excuse me, Help me navigate towards parking ne...",NAVIGATION
1,2,"Assistant, I am not going to electrical on campus",NON_NAVIGATION
2,3,"Assistant, Could you guide me to electronics a...",NAVIGATION
3,4,"Hey, Let's go to parking near MBA block",NAVIGATION
4,5,"Hey assistant, Please navigate me to Near Bank...",NAVIGATION


In [8]:
df.tail(5)

,id,text,intent
443014,443015,I want to cancel the destination change,NON_NAVIGATION
443015,443016,Why is the auditorium closed,NON_NAVIGATION
443016,443017,Who is the head of the department,NON_NAVIGATION
443017,443018,Can you tell me the distance in kilometres as ...,NON_NAVIGATION
443018,443019,How many classrooms are in the mechanical block,NON_NAVIGATION


In [9]:
df.shape

(443019, 3)

In [10]:
print(df["id"].isnull().sum())

0


In [11]:
print(df["text"].isnull().sum())

0


In [12]:
print(df["intent"].isnull().sum())

0


In [13]:
intent_distribution = df["intent"].value_counts()

In [14]:
intent_distribution

,count
intent,
NAVIGATION,280808
NON_NAVIGATION,162211


In [15]:
print("The intent(Navigation) in the overall data is :- " , round((intent_distribution["NAVIGATION"]/(intent_distribution["NAVIGATION"]+intent_distribution["NON_NAVIGATION"])) * 100 , 2))

The intent(Navigation) in the overall data is :-  63.39


In [16]:
print("The intent(Non - Navigation) in the overall data is :- " , round((intent_distribution["NON_NAVIGATION"]/(intent_distribution["NAVIGATION"]+intent_distribution["NON_NAVIGATION"])) * 100 , 2))

The intent(Non - Navigation) in the overall data is :-  36.61


In [17]:
print((df['text'] == '').sum())

0


In [18]:
print(df['text'].duplicated().sum())

0


In [19]:
print(df['text'].str.contains(r'<[^>]+>' , regex = True).sum())

0


In [20]:
print(df['text'].str.contains(r'https?://\S+|www\.\S+' , regex = True).sum())

0


# Text Preprocessing

In [21]:
df['text'] = df['text'].str.lower()

In [22]:
df['text'] = df['text'].replace(r'\s+' , ' ' , regex = True).str.strip()

In [23]:
df.head()

,id,text,intent
0,1,"excuse me, help me navigate towards parking ne...",NAVIGATION
1,2,"assistant, i am not going to electrical on campus",NON_NAVIGATION
2,3,"assistant, could you guide me to electronics a...",NAVIGATION
3,4,"hey, let's go to parking near mba block",NAVIGATION
4,5,"hey assistant, please navigate me to near bank...",NAVIGATION


In [24]:
df['text'] = df['text'].apply(contractions.fix)

In [25]:
df.head()

,id,text,intent
0,1,"excuse me, help me navigate towards parking ne...",NAVIGATION
1,2,"assistant, i am not going to electrical on campus",NON_NAVIGATION
2,3,"assistant, could you guide me to electronics a...",NAVIGATION
3,4,"hey, let us go to parking near mba block",NAVIGATION
4,5,"hey assistant, please navigate me to near bank...",NAVIGATION


# Tokenisation

In [26]:
df['tokens'] = df['text'].apply(lambda x : word_tokenize(x.lower()))

In [27]:
df.head()

,id,text,intent,tokens
0,1,"excuse me, help me navigate towards parking ne...",NAVIGATION,"[excuse, me, ,, help, me, navigate, towards, p..."
1,2,"assistant, i am not going to electrical on campus",NON_NAVIGATION,"[assistant, ,, i, am, not, going, to, electric..."
2,3,"assistant, could you guide me to electronics a...",NAVIGATION,"[assistant, ,, could, you, guide, me, to, elec..."
3,4,"hey, let us go to parking near mba block",NAVIGATION,"[hey, ,, let, us, go, to, parking, near, mba, ..."
4,5,"hey assistant, please navigate me to near bank...",NAVIGATION,"[hey, assistant, ,, please, navigate, me, to, ..."


In [28]:
df.tail()

,id,text,intent,tokens
443014,443015,i want to cancel the destination change,NON_NAVIGATION,"[i, want, to, cancel, the, destination, change]"
443015,443016,why is the auditorium closed,NON_NAVIGATION,"[why, is, the, auditorium, closed]"
443016,443017,who is the head of the department,NON_NAVIGATION,"[who, is, the, head, of, the, department]"
443017,443018,can you tell me the distance in kilometres as ...,NON_NAVIGATION,"[can, you, tell, me, the, distance, in, kilome..."
443018,443019,how many classrooms are in the mechanical block,NON_NAVIGATION,"[how, many, classrooms, are, in, the, mechanic..."


# Exploring the dataset dataframe after preprocessing

In [29]:
print(df['text'].duplicated().sum())

10709


In [30]:
label_counts = df.groupby('text')["intent"].nunique()
conflicts = label_counts[label_counts > 1]
print("Conflicting Sentences :- " , len(conflicts))

Conflicting Sentences :-  0


# Removing unrequired Punctuations

In [31]:
def remove_punctuation(tokens) :
    return[
        word for word in tokens
        if word not in string.punctuation
    ]

In [32]:
df["tokens_clean"] = df['tokens'].apply(remove_punctuation)

In [33]:
df.head()

,id,text,intent,tokens,tokens_clean
0,1,"excuse me, help me navigate towards parking ne...",NAVIGATION,"[excuse, me, ,, help, me, navigate, towards, p...","[excuse, me, help, me, navigate, towards, park..."
1,2,"assistant, i am not going to electrical on campus",NON_NAVIGATION,"[assistant, ,, i, am, not, going, to, electric...","[assistant, i, am, not, going, to, electrical,..."
2,3,"assistant, could you guide me to electronics a...",NAVIGATION,"[assistant, ,, could, you, guide, me, to, elec...","[assistant, could, you, guide, me, to, electro..."
3,4,"hey, let us go to parking near mba block",NAVIGATION,"[hey, ,, let, us, go, to, parking, near, mba, ...","[hey, let, us, go, to, parking, near, mba, block]"
4,5,"hey assistant, please navigate me to near bank...",NAVIGATION,"[hey, assistant, ,, please, navigate, me, to, ...","[hey, assistant, please, navigate, me, to, nea..."


In [34]:
df.tail()

,id,text,intent,tokens,tokens_clean
443014,443015,i want to cancel the destination change,NON_NAVIGATION,"[i, want, to, cancel, the, destination, change]","[i, want, to, cancel, the, destination, change]"
443015,443016,why is the auditorium closed,NON_NAVIGATION,"[why, is, the, auditorium, closed]","[why, is, the, auditorium, closed]"
443016,443017,who is the head of the department,NON_NAVIGATION,"[who, is, the, head, of, the, department]","[who, is, the, head, of, the, department]"
443017,443018,can you tell me the distance in kilometres as ...,NON_NAVIGATION,"[can, you, tell, me, the, distance, in, kilome...","[can, you, tell, me, the, distance, in, kilome..."
443018,443019,how many classrooms are in the mechanical block,NON_NAVIGATION,"[how, many, classrooms, are, in, the, mechanic...","[how, many, classrooms, are, in, the, mechanic..."


# Final Tokenisation Steps

In [35]:

negotiation_words = ['not' , 'no' , 'never' , 'cannot']
for word in negotiation_words :
  count = sum(
      word in tokens for tokens in df['tokens_clean']
  )
  print(word , ':' , count)

not : 63837
no : 2895
never : 0
cannot : 0


In [36]:
destination_words = [
    "admin", "library", "canteen",
    "auditorium", "hostel", "stadium",
    "bank", "electrical", "chemistry"
]
for word in destination_words :
  count = sum(
      word in tokens for tokens in df['tokens_clean']
  )
  print(word , ":" , count)

admin : 7785
library : 15571
canteen : 15568
auditorium : 7784
hostel : 31083
stadium : 30991
bank : 34858
electrical : 11650
chemistry : 11650


In [37]:
empty_tokens = df['tokens_clean'].apply(len).eq(0).sum()
print("Empty Tokens list" , empty_tokens)

Empty Tokens list 0


In [38]:
df['tokens_length'] = df['tokens_clean'].apply(len)
print("Average token length :- " , df['tokens_length'].mean() , 2)

Average token length :-  9.091919308201229 2


In [39]:
df.head()

,id,text,intent,tokens,tokens_clean,tokens_length
0,1,"excuse me, help me navigate towards parking ne...",NAVIGATION,"[excuse, me, ,, help, me, navigate, towards, p...","[excuse, me, help, me, navigate, towards, park...",13
1,2,"assistant, i am not going to electrical on campus",NON_NAVIGATION,"[assistant, ,, i, am, not, going, to, electric...","[assistant, i, am, not, going, to, electrical,...",9
2,3,"assistant, could you guide me to electronics a...",NAVIGATION,"[assistant, ,, could, you, guide, me, to, elec...","[assistant, could, you, guide, me, to, electro...",12
3,4,"hey, let us go to parking near mba block",NAVIGATION,"[hey, ,, let, us, go, to, parking, near, mba, ...","[hey, let, us, go, to, parking, near, mba, block]",9
4,5,"hey assistant, please navigate me to near bank...",NAVIGATION,"[hey, assistant, ,, please, navigate, me, to, ...","[hey, assistant, please, navigate, me, to, nea...",10


In [40]:
df.tail()

,id,text,intent,tokens,tokens_clean,tokens_length
443014,443015,i want to cancel the destination change,NON_NAVIGATION,"[i, want, to, cancel, the, destination, change]","[i, want, to, cancel, the, destination, change]",7
443015,443016,why is the auditorium closed,NON_NAVIGATION,"[why, is, the, auditorium, closed]","[why, is, the, auditorium, closed]",5
443016,443017,who is the head of the department,NON_NAVIGATION,"[who, is, the, head, of, the, department]","[who, is, the, head, of, the, department]",7
443017,443018,can you tell me the distance in kilometres as ...,NON_NAVIGATION,"[can, you, tell, me, the, distance, in, kilome...","[can, you, tell, me, the, distance, in, kilome...",12
443018,443019,how many classrooms are in the mechanical block,NON_NAVIGATION,"[how, many, classrooms, are, in, the, mechanic...","[how, many, classrooms, are, in, the, mechanic...",8


In [41]:
print("Maximum token length :- " , df['tokens_length'].max())

Maximum token length :-  21


In [42]:
print("minimum token length :- ", df['tokens_length'].min())

minimum token length :-  1


In [43]:
df["processed_text"] = df["tokens_clean"].apply(
    lambda tokens: " ".join(tokens)
)

label_counts = df.groupby("processed_text")["intent"].nunique()

conflicts = label_counts[label_counts > 1]

print("Conflicting labels:", len(conflicts))

Conflicting labels: 0


In [44]:
df.head()

,id,text,intent,tokens,tokens_clean,tokens_length,processed_text
0,1,"excuse me, help me navigate towards parking ne...",NAVIGATION,"[excuse, me, ,, help, me, navigate, towards, p...","[excuse, me, help, me, navigate, towards, park...",13,excuse me help me navigate towards parking nea...
1,2,"assistant, i am not going to electrical on campus",NON_NAVIGATION,"[assistant, ,, i, am, not, going, to, electric...","[assistant, i, am, not, going, to, electrical,...",9,assistant i am not going to electrical on campus
2,3,"assistant, could you guide me to electronics a...",NAVIGATION,"[assistant, ,, could, you, guide, me, to, elec...","[assistant, could, you, guide, me, to, electro...",12,assistant could you guide me to electronics an...
3,4,"hey, let us go to parking near mba block",NAVIGATION,"[hey, ,, let, us, go, to, parking, near, mba, ...","[hey, let, us, go, to, parking, near, mba, block]",9,hey let us go to parking near mba block
4,5,"hey assistant, please navigate me to near bank...",NAVIGATION,"[hey, assistant, ,, please, navigate, me, to, ...","[hey, assistant, please, navigate, me, to, nea...",10,hey assistant please navigate me to near bank ...


In [45]:
df.shape

(443019, 7)

In [46]:
df = df.drop_duplicates(
    subset = ['processed_text'],
    keep = 'first'
).reset_index(drop = True)

In [47]:
df.head()

,id,text,intent,tokens,tokens_clean,tokens_length,processed_text
0,1,"excuse me, help me navigate towards parking ne...",NAVIGATION,"[excuse, me, ,, help, me, navigate, towards, p...","[excuse, me, help, me, navigate, towards, park...",13,excuse me help me navigate towards parking nea...
1,2,"assistant, i am not going to electrical on campus",NON_NAVIGATION,"[assistant, ,, i, am, not, going, to, electric...","[assistant, i, am, not, going, to, electrical,...",9,assistant i am not going to electrical on campus
2,3,"assistant, could you guide me to electronics a...",NAVIGATION,"[assistant, ,, could, you, guide, me, to, elec...","[assistant, could, you, guide, me, to, electro...",12,assistant could you guide me to electronics an...
3,4,"hey, let us go to parking near mba block",NAVIGATION,"[hey, ,, let, us, go, to, parking, near, mba, ...","[hey, let, us, go, to, parking, near, mba, block]",9,hey let us go to parking near mba block
4,5,"hey assistant, please navigate me to near bank...",NAVIGATION,"[hey, assistant, ,, please, navigate, me, to, ...","[hey, assistant, please, navigate, me, to, nea...",10,hey assistant please navigate me to near bank ...


In [48]:
print(df['text'].duplicated().sum())

0


In [49]:
print(df['processed_text'].duplicated().sum())

0


In [50]:
df.shape

(429574, 7)

# Spliting the data into train , validation and test

In [51]:
X = df['processed_text']
y = df['intent']

X_train , X_test , y_train , y_test = train_test_split(
    X , y , test_size = 0.2 , random_state = 42 , stratify = y
)

In [52]:
X_train.shape

(343659,)

In [53]:
X_test.shape

(85915,)

In [54]:
y_train.shape

(343659,)

In [55]:
y_test.shape

(85915,)

In [56]:
X_train , X_val , y_train , y_val = train_test_split(
    X_train , y_train , test_size = 0.1 , random_state = 42 , stratify = y_train
)

In [57]:
X_train.shape

(309293,)

In [58]:
X_val.shape

(34366,)

In [59]:
y_train.shape

(309293,)

In [60]:
y_val.shape

(34366,)

# Converting all the words of each sentence into a corpus

In [61]:
sentences = X_train.apply(lambda x : x.split()).to_list()

In [62]:
print(sentences[0])
print(sentences[2])
print(sentences[4])
print(sentences[6])
print(sentences[8])
print(sentences[10])

['actually', 'is', 'this', 'near', 'coffee', 'shop', 'parking', 'please']
['actually', 'i', 'am', 'waiting', 'at', 'biotechnology', 'block', 'right', 'now']
['find', 'a', 'path', 'to', 'bvb', 'hostel', 'right', 'now']
['excuse', 'me', 'directions', 'to', 'mba', 'department', 'at', 'sit']
['hey', 'please', 'do', 'not', 'take', 'me', 'to', 'computer', 'science', 'department']
['assistant', 'i', 'am', 'looking', 'at', 'auditorium', 'on', 'campus']


# Applying Word to vec

In [65]:
word2vec_model = Word2Vec(
    sentences = sentences ,
    vector_size = 100 ,
    window = 5 ,
    min_count = 2 ,
    sg = 1 ,
    workers = 4 ,
    epochs = 10 ,
    seed = 42,
    compute_loss = True
)

In [66]:
print("Vocabulary size :- " , len(word2vec_model.wv))
print("Vector for admin :- " , word2vec_model.wv['admin'])
print("Vector for library :- " , word2vec_model.wv['library'])
print("Vector for coffee :- " , word2vec_model.wv['coffee'])
print("Vector for canteen :- " , word2vec_model.wv['canteen'])

Vocabulary size :-  279
Vector for admin :-  [-0.07470512 -0.06561541  0.31153387  0.2262009   0.10921784 -0.08498664
 -0.0217061  -0.22531013 -0.02141591  0.09840377 -0.00632522 -0.01620634
  0.28338408  0.30536345 -0.03546886  0.05693395  0.23235875  0.2057894
  0.2759688   0.02878007  0.11521466 -0.02535272  0.6845624   0.16600268
  0.18040888  0.38620692 -0.29184926  0.13345753 -0.09784897 -0.2433805
 -0.406449   -0.00622309 -0.21410574 -0.14040472 -0.2544867  -0.07827917
  0.26441678  0.1228475   0.08109877 -0.16801265  0.17624804  0.1972353
  0.38443366  0.10359622  0.08169529  0.08669766  0.19068366 -0.36638597
  0.43246984  0.07063317 -0.06492574 -0.521158   -0.33444047  0.14713036
 -0.03808514 -0.2379068   0.16745558  0.02176012 -0.2365708  -0.29748684
 -0.11490131  0.07569766 -0.57592005 -0.1181265   0.9933995  -0.22522388
 -0.16714112 -0.38121954 -0.1934056   0.22279274 -0.1402207  -0.12849107
 -0.36364576  0.03928105  0.14426799 -0.04070781 -0.3569931  -0.07454295
 -0.22876

# Giving each word a index instead of representing it in 100 x 1 form

In [67]:
word_to_index = {
    word : index + 1
    for index , word in enumerate(word2vec_model.wv.index_to_key)
}

In [68]:
print("vocabulary_size :- " , len(word_to_index))

vocabulary_size :-  279


# Converting each sentence into its word to index form

In [69]:
def convert_to_sequences (data) :
  sequences = []
  for sentence in data :
    words = sentence.split()
    sequence = []
    for word in words :
      if word in word_to_index :
        index = word_to_index[word]
        sequence.append(index)
    sequences.append(sequence)
  return sequences

In [70]:
print(X_train.iloc[0])
print(X_val.iloc[0])
print(X_test.iloc[0])

actually is this near coffee shop parking please
please i want directions for biotechnology department right now
actually i am not asking to go to mechanical department please


In [71]:
X_train_seq = convert_to_sequences(X_train)
X_val_seq = convert_to_sequences(X_val)
X_test_seq = convert_to_sequences(X_test)

In [72]:
print(X_train_seq[0])
print(X_val_seq[0])
print(X_test_seq[0])

[32, 29, 177, 27, 60, 59, 15, 3]
[3, 4, 39, 36, 87, 104, 26, 11, 7]
[32, 4, 24, 14, 118, 1, 16, 1, 71, 26, 3]


In [73]:
# Find the length of each sentence

train_lengths = []

for sequence in X_train_seq:
    train_lengths.append(len(sequence))

print("Shortest sentence:", min(train_lengths))
print("Longest sentence:", max(train_lengths))

print("Average length:", sum(train_lengths) / len(train_lengths))

Shortest sentence: 0
Longest sentence: 21
Average length: 9.063790645116443


# Applying Padding

In [74]:
max_len = 21
X_train_pad = pad_sequences(
    X_train_seq ,
    maxlen = max_len ,
    padding = 'post' ,
    truncating = 'post'
)
X_val_pad = pad_sequences(
    X_val_seq ,
    maxlen = max_len ,
    padding = 'post' ,
    truncating = 'post'
)
X_test_pad = pad_sequences(
    X_test_seq ,
    maxlen = max_len ,
    padding = 'post' ,
    truncating = 'post'
)

In [75]:
X_train_pad.shape

(309293, 21)

In [76]:
X_test_pad.shape

(85915, 21)

In [77]:
X_val_pad.shape

(34366, 21)

# Creating Embedding matrix each senctence ----> indivusal words ----> vector form

In [78]:
vocab_size = len(word2vec_model.wv)
embedding_matrix = np.zeros((vocab_size + 1 , 100))
for index , word in enumerate(word2vec_model.wv.index_to_key):
  embedding_matrix[index + 1] = word2vec_model.wv[word]

In [79]:
print("Voabulary_size :- " , vocab_size)
print("Embedding matrix shape :- " , embedding_matrix.shape)

Voabulary_size :-  279
Embedding matrix shape :-  (280, 100)


In [80]:
embedding_matrix[1]

array([-0.0642757 , -0.01958559,  0.0371862 ,  0.36202326, -0.18889011,
       -0.05195884,  0.38250852, -0.00300855, -0.23082694, -0.19422345,
       -0.43737367,  0.04932901,  0.02968528, -0.0813264 , -0.11655492,
        0.19995923, -0.30111974,  0.28789851,  0.24712083, -0.08660759,
       -0.05556541,  0.20269196, -0.00497704, -0.58383185,  0.17922701,
       -0.0913533 ,  0.02722177, -0.16502522,  0.0666275 , -0.17638361,
       -0.03283241, -0.42921665, -0.05198878, -0.31041661,  0.61422354,
        0.15903385,  0.17566897, -0.0016163 ,  0.03598379, -0.3158913 ,
        0.02836559,  0.02728718, -0.21547492, -0.05036321,  0.01757049,
        0.23501745,  0.24191487,  0.28600663,  0.3383511 ,  0.02232616,
       -0.12737776,  0.1782085 , -0.16990525, -0.23309685, -0.42495388,
        0.0953892 , -0.10315587,  0.16234083, -0.29411873, -0.22961947,
        0.7756111 ,  0.31268975,  0.1451946 , -0.07289118, -0.11862225,
       -0.09039232,  0.02334805,  0.24080648, -0.16940026, -0.16

In [ ]:
X_train_embedded = embedding_matrix[X_train_pad]
X_val_embedded = embedding_matrix[X_val_pad]
X_test_embedded = embedding_matrix[X_test_pad]

In [ ]:
X_train_embedded[0]

In [81]:
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

In [82]:
print(y_train_encoded[0])

1


In [ ]:
del X_train_embedded
del X_val_embedded
del X_test_embedded

import gc
gc.collect()

In [83]:
gru_model = Sequential()
gru_model.add(Input(shape = (max_len , )))
gru_model.add(
    Embedding(
        input_dim = vocab_size + 1 ,
        output_dim = 100 ,
        weights = [embedding_matrix] ,
        mask_zero = True ,
        trainable = False
    )
)
gru_model.add(GRU(64))
gru_model.add(Dense(1 , activation = 'sigmoid'))

In [84]:
gru_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 21, 100)        │        28,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │        31,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 59,937 (234.13 KB)

 Trainable params: 31,937 (124.75 KB)

 Non-trainable params: 28,000 (109.38 KB)

In [85]:
gru_model.compile(optimizer = 'adam' , loss = 'binary_crossentropy' , metrics = ['accuracy'])

In [86]:
early_stop = EarlyStopping(
    monitor = 'val_loss' ,
    patience = 2 ,
    verbose = 1 ,
    restore_best_weights = True
)

In [87]:
history1 = gru_model.fit(
    X_train_pad,
    y_train_encoded,
    validation_data = (X_val_pad , y_val_encoded),
    epochs = 10 ,
    batch_size = 32 ,
    callbacks = [early_stop] ,
    verbose = 1
)

Epoch 1/10
9666/9666 ━━━━━━━━━━━━━━━━━━━━ 148s 15ms/step - accuracy: 0.9985 - loss: 0.0037 - val_accuracy: 0.9999 - val_loss: 4.7389e-04
Epoch 2/10
9666/9666 ━━━━━━━━━━━━━━━━━━━━ 145s 15ms/step - accuracy: 0.9999 - loss: 3.0181e-04 - val_accuracy: 1.0000 - val_loss: 3.0989e-04
Epoch 3/10
9666/9666 ━━━━━━━━━━━━━━━━━━━━ 146s 15ms/step - accuracy: 1.0000 - loss: 1.0024e-04 - val_accuracy: 1.0000 - val_loss: 3.5027e-04
Epoch 4/10
9666/9666 ━━━━━━━━━━━━━━━━━━━━ 146s 15ms/step - accuracy: 1.0000 - loss: 6.6542e-05 - val_accuracy: 1.0000 - val_loss: 3.0270e-04
Epoch 5/10
9666/9666 ━━━━━━━━━━━━━━━━━━━━ 146s 15ms/step - accuracy: 1.0000 - loss: 7.7701e-05 - val_accuracy: 1.0000 - val_loss: 4.0524e-04
Epoch 6/10
9666/9666 ━━━━━━━━━━━━━━━━━━━━ 208s 16ms/step - accuracy: 1.0000 - loss: 4.1428e-05 - val_accuracy: 1.0000 - val_loss: 3.9770e-04
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 4.


In [88]:
gru_model.save('/content/drive/MyDrive/smart blind stick/models/gru_word2vec_model.keras')

In [91]:
bilstm_model = Sequential()
bilstm_model.add(Input(shape = (max_len , )))
bilstm_model.add(
    Embedding(
        input_dim = vocab_size + 1 ,
        output_dim = 100 ,
        weights = [embedding_matrix] ,
        mask_zero = True ,
        trainable = False
    )
)
bilstm_model.add(Bidirectional(LSTM(64)))
bilstm_model.add(Dense(1 , activation = 'sigmoid'))

In [92]:
bilstm_model.compile(
    optimizer = 'adam' ,
    loss = 'binary_crossentropy' ,
    metrics = ['accuracy']
)

In [93]:
bilstm_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 21, 100)        │        28,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 128)            │        84,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 112,609 (439.88 KB)

 Trainable params: 84,609 (330.50 KB)

 Non-trainable params: 28,000 (109.38 KB)

In [94]:
history2 = bilstm_model.fit(X_train_pad ,
                            y_train_encoded ,
                            validation_data = (X_val_pad , y_val_encoded),
                            epochs = 10 ,
                            batch_size = 32 ,
                            callbacks = [early_stop],
                            verbose = 1
)

Epoch 1/10
9666/9666 ━━━━━━━━━━━━━━━━━━━━ 252s 26ms/step - accuracy: 0.9989 - loss: 0.0027 - val_accuracy: 1.0000 - val_loss: 4.1810e-04
Epoch 2/10
9666/9666 ━━━━━━━━━━━━━━━━━━━━ 250s 26ms/step - accuracy: 0.9999 - loss: 2.0070e-04 - val_accuracy: 1.0000 - val_loss: 3.8650e-04
Epoch 2: early stopping
Restoring model weights from the end of the best epoch: 1.


In [96]:
bilstm_model.save('/content/drive/MyDrive/smart blind stick/models/bilstm_word2vec_model.keras')

In [97]:
gru_predictions = gru_model.predict(X_test_pad , batch_size = 128)

672/672 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step


In [98]:
bilstm_predictions = bilstm_model.predict(X_test_pad , batch_size = 128)

672/672 ━━━━━━━━━━━━━━━━━━━━ 16s 23ms/step


In [99]:
# Convert GRU probabilities into class labels
gru_predictions = (gru_predictions >= 0.5).astype(int).ravel()

# Convert BiLSTM probabilities into class labels
bilstm_predictions = (bilstm_predictions >= 0.5).astype(int).ravel()

In [100]:
print("GRU Classification Report")

print(classification_report(
    y_test_encoded,
    gru_predictions,
    target_names=label_encoder.classes_
))

GRU Classification Report
                precision    recall  f1-score   support

    NAVIGATION       1.00      1.00      1.00     54841
NON_NAVIGATION       1.00      1.00      1.00     31074

      accuracy                           1.00     85915
     macro avg       1.00      1.00      1.00     85915
  weighted avg       1.00      1.00      1.00     85915



In [101]:
print("BiLSTM Classification Report")

print(classification_report(
    y_test_encoded,
    bilstm_predictions,
    target_names=label_encoder.classes_
))

BiLSTM Classification Report
                precision    recall  f1-score   support

    NAVIGATION       1.00      1.00      1.00     54841
NON_NAVIGATION       1.00      1.00      1.00     31074

      accuracy                           1.00     85915
     macro avg       1.00      1.00      1.00     85915
  weighted avg       1.00      1.00      1.00     85915



In [102]:
from sklearn.metrics import accuracy_score

gru_accuracy = accuracy_score(
    y_test_encoded,
    gru_predictions
)

bilstm_accuracy = accuracy_score(
    y_test_encoded,
    bilstm_predictions
)

print("GRU Test Accuracy:", round(gru_accuracy * 100, 2), "%")

print("BiLSTM Test Accuracy:", round(bilstm_accuracy * 100, 2), "%")

GRU Test Accuracy: 100.0 %
BiLSTM Test Accuracy: 100.0 %


In [103]:
with open('/content/drive/MyDrive/smart blind stick/models/word_to_index.json', 'w') as f:
  json.dump(word_to_index , f)
print("Vocabulary mapping saved!")

Vocabulary mapping saved!
